In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/032026/Data_OOT/MEDS_MDPS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,12,NaT,GENDER//Kvinde,NaN
1,12,1945-07-05 00:00:00,DOB,NaN
2,12,2017-03-16 00:00:00,D/DZ950B,NaN
3,12,2017-03-16 14:00:00,P/ZZ0150,NaN
4,12,2017-03-16 14:46:00,P/ZZ3925,NaN
5,12,2017-03-22 07:57:00,ADMISSION_ADT,NaN
6,12,2017-03-22 07:57:00,DISCHARGE_ADT,NaN
7,12,2017-03-22 07:57:00,MOVE_ADT,NaN
8,12,2017-03-22 07:57:00,^AFSNIT_ADT/,NaN
9,12,2017-03-22 09:01:00,P/ZZ3925,NaN


In [2]:
len(df)

588438377

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 2218028
The patients has M-medication Codes: 1511054
The patients has D-diagnosis Codes: 2217421
The patients has P-Procedure Codes: 2126853
The patients has S-SKS Codes: 526005


In [4]:
subject_counts = df['subject_id'].value_counts()

In [5]:
subject_counts

921542     82180
106        63835
698589     63448
2016077    60882
954669     58832
           ...  
471874         2
80783          2
85775          2
1682596        2
131767         2
Name: subject_id, Length: 2218028, dtype: int64

In [6]:
s_Num = df[df['code'].str.startswith('S/', na=False)]

In [7]:
s_Num

,subject_id,time,code,numeric_value
157,12,2021-07-20 23:59:00,S/KNCJ65,NaN
669,147,2017-06-18 23:59:00,S/KCJE20,NaN
1256,290,2019-06-03 23:59:00,S/KFNG05,NaN
1961,303,2021-04-28 23:59:00,S/KABD60B,NaN
3171,518,2017-07-17 23:59:00,S/KCKE60,NaN
...,...,...,...,...
588434691,2217262,2019-09-19 23:59:00,S/KNFJ51,NaN
588435336,2217307,2019-04-02 23:59:00,S/KQBE10B,NaN
588436199,2217362,2021-05-28 23:59:00,S/KNHK97,NaN
588437010,2217560,2019-09-02 23:59:00,S/KCJE20,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Srugery code: ", only_p_ids_to_exclude)


Number of patients with only Srugery code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

921542     82176
106        63833
698589     63448
2016077    60882
954669     58832
           ...  
484192         2
131767         2
1703144        2
85775          2
1434427        2
Name: subject_id, Length: 2218028, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
14786,1719866,3685,3656,29
20156,312261,3158,3129,29
8980,1541863,4646,4618,28
60130,2162677,1686,1658,28
2188,1281561,8406,8378,28
...,...,...,...,...
1016510,1281602,92,92,0
1016509,767419,92,92,0
1016508,1394144,92,92,0
1016507,2090469,92,92,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 1692023


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
14786     1719866            3685       3656          29        29
20156      312261            3158       3129          29        29
8980      1541863            4646       4618          28        28
60130     2162677            1686       1658          28        28
2188      1281561            8406       8378          28        28
9266       121479            4582       4556          26        26
11771      736003            4107       4082          25        25
1637      1966150            9429       9404          25        25
24459     1931799            2860       2836          24        24
99785      712426            1192       1170          22        22


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [22]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDP codes'
    }
)


In [23]:
lowest_new_count_patients

,subject_id,MDPS codes,MDP codes,difference,abs_diff
2157755,228541,6,5,1,1
2153272,1660625,6,5,1,1
2151999,1864799,6,5,1,1
2150128,451630,6,5,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [24]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

2218028

In [25]:
len(df_filtered)

587671613

In [26]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 587671613  / total: 588438377


In [27]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 587671613


In [28]:
import numpy as np
import os

N_SHARDS = 45    #45 for Whole # 36 when we have split
OUT_DIR = "./_TrainMDPS_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 45 parquet files into ./_TrainMDPS_withoutS_sharded


In [28]:
import pyarrow.parquet as pq

OUT_DIR = "./_TrainMDPS_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 12347793
1.parquet rows: 12492159
10.parquet rows: 12061876
11.parquet rows: 12409035
12.parquet rows: 12297708
13.parquet rows: 12533268
14.parquet rows: 12457981
15.parquet rows: 12421926
16.parquet rows: 12341002
17.parquet rows: 12244372
18.parquet rows: 12526692
19.parquet rows: 12759747
2.parquet rows: 12572763
20.parquet rows: 12447479
21.parquet rows: 12541163
22.parquet rows: 12690784
23.parquet rows: 12504444
24.parquet rows: 12280464
25.parquet rows: 12459476
26.parquet rows: 12539646
27.parquet rows: 12340156
28.parquet rows: 12697448
29.parquet rows: 12212392
3.parquet rows: 12317699
30.parquet rows: 12268776
31.parquet rows: 12474665
32.parquet rows: 12575590
33.parquet rows: 12459496
34.parquet rows: 12140644
35.parquet rows: 12447176
4.parquet rows: 12211682
5.parquet rows: 12791078
6.parquet rows: 12525623
7.parquet rows: 12264458
8.parquet rows: 12642153
9.parquet rows: 12591599
TOTAL rows: 447890413


In [29]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_TrainMDPS_withoutS_sharded"
DST_PREFIX = "Zahra/032026/Data_OOT/MEDS_MDP/data/train"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 45
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutS_sharded' to 'Zahra/032026/Data_OOT/MEDS_MDP/data/train'
Copying 45 files with concurrency set to 6
Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutS_sharded/12.parquet, file 1 out of 45. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/032026/Data_OOT/MEDS_MDP/data/train/12.parquet
Copied /mnt/b

In [30]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 45
['/0.parquet', '/1.parquet', '/10.parquet', '/11.parquet', '/12.parquet', '/13.parquet', '/14.parquet', '/15.parquet', '/16.parquet', '/17.parquet', '/18.parquet', '/19.parquet', '/2.parquet', '/20.parquet', '/21.parquet', '/22.parquet', '/23.parquet', '/24.parquet', '/25.parquet', '/26.parquet']
